In [0]:
import nltk
nltk.download('stopwords')

In [0]:
from pyspark.sql import SparkSession, functions as F

from pyspark.ml import Pipeline
from pyspark.ml.feature import RegexTokenizer, StopWordsRemover, CountVectorizer, IDF
from pyspark.ml.clustering import KMeans, BisectingKMeans
from pyspark.ml.evaluation import ClusteringEvaluator

from nltk.corpus import stopwords

In [0]:
spark = SparkSession.builder.getOrCreate()

In [0]:
df = (
    spark
    .read
    .table('dados_fa.pessoal.org_xml')
    .select(['arquivo_origem', 'texto', 'artType'])
    .filter('texto is not null')
)

In [0]:
df_base = (
    df
    .withColumn('texto', F.lower(F.col('texto')))
    .withColumn('texto', F.regexp_replace('texto', r'\d+', ' '))
    .withColumn('texto', F.regexp_replace('texto', r'\s+', ' '))
)

tokenizer = RegexTokenizer(
    inputCol='texto',
    outputCol='tokens', 
    pattern=r'\W+',
    toLowercase=True
)

stop_words = stopwords.words('portuguese') + ['p', 'class', 'pdf', 'n', 'td', 'justify', 'corpo', 'colspan', 'rowspan', 's', 'art', 'es', 'ncia', 'rio', 'r', 'tr', 'inciso', 'c', 'identifica', 'lei', 'portaria', 'processo', 'valor', 'cpf', 'objeto', 'data', 'ltda', 'i', 'total', 'br', 'ii', 'servi', 'center', 'm'] + ['n', 'td', 'corpo', 'colspan', 'rowspan', 's', 'art', 'es', 'ncia', 'rio', 'r', 'tr', 'h', 'inciso', 'c', 'lei', 'portaria', 'processo', 'valor', 'data', 'ltda', 'i', 'br', 'total', 'ii', 'servi', 'm', 't', 'preg', 'minist', 'ex', 'ne', 'legal', 'nico', 'fundamento', 'acordo', 'vig', 'ria', 'cargo', 'dezembro', 'edital', 'licita', 'tcu', 'www']

remover = StopWordsRemover(
    inputCol='tokens', 
    outputCol='tokens_filtrados',
    stopWords=stop_words
)

vectorizer = CountVectorizer(
    inputCol='tokens_filtrados', 
    outputCol='tf_features', 
    vocabSize=5000,
    minDF=1.0
)

idf = IDF(
    inputCol='tf_features', 
    outputCol='features'
)

kmeans = KMeans(
    featuresCol='features', 
    predictionCol='cluster', 
    k=25, 
    seed=42
)

pipeline = Pipeline(stages=[tokenizer, remover, vectorizer, idf, kmeans])

modelo = pipeline.fit(df_base)
resultado = modelo.transform(df_base)

In [0]:
resultado.select(['arquivo_origem', 'artType', 'cluster']).show(truncate=False)

In [0]:
cv_model = modelo.stages[2]
vocabulario = cv_model.vocabulary

print('Tamanho do vocabulário:', len(vocabulario))
print(vocabulario[:50])

In [0]:
resultado.orderBy('cluster', 'arquivo_origem').select('cluster', 'arquivo_origem', 'artType').show(truncate=False)

In [0]:
resultado.groupBy('cluster', 'tokens_filtrados').count().orderBy('cluster', 'count', ascending=False).display()

In [0]:
resultado.show()

In [0]:
resultado.select('tokens_filtrados').display()